![Astrofisica Computacional](../../../new_logo.png)

## Dr. rer. nat. Jose Ivan Campos Rozo<sup>1,2</sup>

1. Hvar Observatory\
    Faculty of Geodesy, University of Zagreb\
    Zagreb, Croatia

2. Observatorio Astronómico Nacional\
   Facultad de Ciencias\
   Universidad Nacional de Colombia

e-mail: jicamposr@unal.edu.co & jcamposro@geof.hr)

---

# NumPy

Ejercicios avanzados de NumPy + Funciones/Clases + Archivos

**Conjunto de datos:** Adaptado del conjunto de datos de temperaturas de Helsinki de 2017 (https://raw.githubusercontent.com/csmastersUH/data_analysis_with_python_2020/master/kumpula-weather-2017.csv)

Incluye:
- NumPy: segmentación (*slicing*), máscaras, ajuste polinómico (*polyfit*), FFT.

- Funciones/Clases: herencia, decoradores, métodos vectorizados.

- Archivos: `loadtxt`, `savetxt` con datos procesados.

In [82]:
import numpy as np

data = np.genfromtxt("kumpula-weather-2017.csv", delimiter=",", dtype=float, names=True, usecols=(1,2,5,6,7))

## Ejercicio 1: Carga de datos + Preprocesamiento con NumPy

Carga las temperaturas desde el archivo "kumpula-weather-2017.csv". Escribe una función para convertir el formato mm-dd en días transcurridos desde el inicio (days = monthday_to_days).

- Filtra los valores NaN, inf o masks.

- Calcula las anomalías (temperatura - media móvil de 30 días).

- Detecta valores atípicos (método del IQR: Q1 - 1.5 * IQR).

**Resultado esperado:** `temps_clean` (N x 2), `days` (N,).

In [150]:
# Función para convertir formato mm-dd a días

# (1A) La función admitirá una función y espera que tenga una columna "m" (month) y "d" (day)

def monthday_to_days(M):

    M_dayformat = M[[cols for cols in M.dtype.names if cols != "m"]]

    for i in range(len(M_dayformat)):
        M_dayformat[i]["d"] = i + 1

    return M_dayformat

data_dayformat = monthday_to_days(data)

# (1B) Ahora buscamos filtrar valores infinitos y NaN. Esto puede hacerse con masks

def filter_invalid(M):

    # Creamos el mask
    mask = np.ones(len(M), dtype=bool)   # Un "True" inicial para todas las columnas, en primera instancia

    for col in M.dtype.names:

         # Para cada columna, hacemos la operación "^" entre la lista de "Trues" y aquella resultante de np.isfinite en la columna actual
         # Así, sólo sobrevive el elemento (la fila) para la cual el elemento es finito (ni NaN ni inf)

        mask &= np.isfinite(M[col])    

        # Al terminar, sólo quedan con "True" las filas que nunca tuvieron un valor no finito

    M_filtered = M[mask]

    return M_filtered

data_dayformat_filtered = filter_invalid(data_dayformat)

# (1C) La media móvil es el promedio de los 30 últimos valores (incluyendo el actual)

def anomalies(M, col):

    anomalies = np.zeros(len(M))

    for i in range(29, len(M)):
        anomalies[i] = M[i][col] - ( np.sum(M[i-29:i+1][col]) / 30 )

    return anomalies

temp_anomalies = anomalies(data_dayformat_filtered, "Air_temperature_degC")

# (1D) Para detectar los valores atípicos con el método IQR

def IQR(M, col):

    column = M[col]
    days = M["d"]

    # Calcular percentiles
    Q1 = np.percentile(column, 25)
    Q3 = np.percentile(column, 75)

    IQR = Q3 - Q1

    # Aplicamos Q1 - 1.5*IQR calculando ambos límites del intervalo:

    lim_inf = Q1 - 1.5*IQR
    lim_sup = Q3 + 1.5*IQR

    # Para detectar las temperaturas que caigan dentro:

    mask_outlier = (column < lim_inf) | (lim_sup < column)
    mask_clean  = ~mask_outlier

    M_outlier = np.column_stack((M["d"][mask_outlier], M[col][mask_outlier]))
    M_clean = np.column_stack((M["d"][mask_clean], M[col][mask_clean]))

    return M_outlier, M_clean

temps_outlier, temps_clean = IQR(data_dayformat_filtered, "Air_temperature_degC")

# Así, los datos atípicos de temperatura son "temp_outlier"

print(np.shape(temps_outlier))

print(np.shape(temps_clean))

days = temps_clean[:,0]
temps = temps_clean[:,1]

print(len(days))


(2, 2)
(356, 2)
356


## Ejercicio 2: Clase avanzada con herencia y decoradores

Crea una **clase base `TimeSeriesAnalyzer`**:
- `smooth(self, window=7)`: media móvil.

- Decorador `@vectorize` para aplicar funciones a las columnas.

La **clase derivada `WeatherAnalyzer`** hereda y añade:
- `seasonal_decompose(self)`: tendencia (ajuste polinómico de grado 2), componente estacional (FFT, 4 frecuencias principales) y residuo.

- `forecast(self, days_ahead=30)`: pronóstico simple tipo ARIMA (últimos 30 días, ajuste polinómico de grado 3) + ruido.

- **Uso:** `analyzer = WeatherAnalyzer(days, temps_clean[:,1])`

In [151]:
# Decorator to vectorize method over the columns
def vectorize(method):
    def wrapper(self, *args, **kwargs):
        results = np.array([method(self, col, *args, **kwargs) for col in self.data.T]).T
        return results if results.ndim > 1 else results.flatten()
    return wrapper

class TimeSeriesAnalyzer:
    """Clase base para análisis de series temporales con NumPy."""
    def __init__(self, t: np.ndarray, data: np.ndarray):
        self.t = np.asarray(t)
        self.data = np.asarray(data)
        if self.t.shape[0] != self.data.shape[0]:
            raise ValueError("t y data deben tener misma longitud")
    
    @vectorize
    def smooth(self, col: np.ndarray, window: int = 7, mode: str = 'same') -> np.ndarray:
        """Media móvil con convolve."""
        kernel = np.ones(window) / window
        return np.convolve(col, kernel, mode=mode)
    
    def stats(self) -> dict:
        """Estadísticas básicas por columna."""
        return {
            'mean': np.mean(self.data, axis=0),
            'std': np.std(self.data, axis=0),
            'min': np.min(self.data, axis=0),
            'max': np.max(self.data, axis=0)
        }

class WeatherAnalyzer(TimeSeriesAnalyzer):
    """Hija especializada en datos meteorológicos."""
    def __init__(self, t: np.ndarray, data: np.ndarray):
        super().__init__(t, data)
    
    def seasonal_decompose(self, degree_trend: int = 2, n_freqs: int = 4) -> dict:
        """Descomposición: trend (polyfit), seasonal (FFT top freqs), residual."""
        # Trend: polyfit global
        p_trend = np.polynomial.Polynomial.fit(self.t, self.data, degree_trend)
        trend = p_trend(self.t)
        
        # Seasonal: FFT, top n_freqs armónicos
        fft = np.fft.fft(self.data - trend, axis=0)
        freqs = np.fft.fftfreq(len(self.t))
        top_idx = np.argsort(np.abs(fft), axis=0)[-n_freqs:][::-1]
        seasonal = np.zeros_like(self.data)
        for idx in top_idx[:]:
            seasonal[:] += 2 * np.real(np.fft.ifft(fft[:] * (np.abs(fft[idx]) > 1e-3)))
        
        residual = self.data - trend - seasonal
        return {'trend': trend, 'seasonal': seasonal, 'residual': residual}
    
    def forecast(self, days_ahead: int = 30, degree: int = 3, noise_std: float = 1.0) -> tuple:
        """Pronóstico simple: polyfit últimos datos + ruido gaussiano."""
        n_last = min(degree * 10, len(self.t) // 2)
        t_last = self.t[-n_last:]
        data_last = self.data[-n_last:]
        
        t_future = np.linspace(self.t[-1], self.t[-1] + days_ahead, days_ahead)
        forecast = np.zeros(days_ahead) #np.zeros((days_ahead, self.data.shape[1]))
        
        p = np.polyfit(t_last, data_last[:], degree)
        forecast[:] = np.polyval(p, t_future) + np.random.normal(0, noise_std, days_ahead)
        
        return t_future, forecast

# DEMO de USO (¡ejecuta después de Ej.1!)
analyzer = WeatherAnalyzer(days, temps)
smoothed = analyzer.smooth(window=15)
decomp = analyzer.seasonal_decompose()
t_fc, fc = analyzer.forecast(30)
print(analyzer.stats())

{'mean': np.float64(6.653932584269663), 'std': np.float64(7.019020005538647), 'min': np.float64(-12.8), 'max': np.float64(19.6)}


## Ejercicio 3: I/O robusta + datos procesados

- Guarda `temps_clean` y `anomalies` en el archivo 'processed_weather.npz' (usando `np.savez`).

- Guarda un subconjunto (primeros 100 días; columnas: `days`, `temp_smooth`, `anomaly`) en 'subset.csv' (usando `savetxt`, con `fmt='%.2f'` y encabezado).

- Función `load_and_validate(filename)`: carga el archivo npz/csv, verifica las dimensiones y la presencia de valores infinitos o NaN, y devuelve un diccionario.

In [ ]:
# np.savez('processed_weather.npz', temps=temps_clean, anomalies=anomalies)
# subset = np.column_stack([dias[:100], smoothed[0,:100], anomalies[:100]])
# np.savetxt('subset.csv', subset, delimiter=',', header='day,temp_smooth,anomaly', fmt='%.3f')

def load_and_validate(filename):
    # Maneja .npz (load), .csv; check np.isfinite.all(), shape==(?,3)
    pass

# Algunas preguntas:

- Para los valores atípicos (método del IQR): ¿qué fracción de los datos se elimina? ¿Es un enfoque conservador o agresivo? Proponga una alternativa (p. ej., sigma=3).
- Analice la ejecución de `@vectorize`: para `data.shape=(365,2)`, ¿cuántas llamadas se realizan a `smooth(col)`? ¿Por qué `results.T`? ¿Qué otro método de vectorización propone?
- En `seasonal_decompose()`: ¿por qué se resta la tendencia antes de la FFT? ¿Qué representan las 4 frecuencias principales (diaria/semanal/mensual/anual)?
- Ejecute `analyzer.stats()` comparando los datos originales frente a los suavizados: ¿cuál es el porcentaje de reducción de la desviación estándar por columna? ¿Por qué `polyfit` de grado 2 captura bien la tendencia?